In [ ]:

import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
import os
import time

# NVD API endpoint
API_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"

# Optional: Set your NVD API key as an environment variable
API_KEY = os.getenv("NVD_API_KEY")

def get_cvss(cve):
    """Extract the highest available CVSS version."""
    metrics = cve.get("metrics", {})

    for version in ["cvssMetricV40", "cvssMetricV31",
                    "cvssMetricV30", "cvssMetricV2"]:
        if version in metrics and metrics[version]:
            data = metrics[version][0].get("cvssData", {})
            return (
                data.get("baseScore", "N/A"),
                data.get("baseSeverity",
                         metrics[version][0].get("baseSeverity", "N/A"))
            )

    return "N/A", "N/A"


def scrape_cves():
    # Current UTC time and five days ago
    end_date = datetime.now(timezone.utc)
    start_date = end_date - timedelta(days=5)

    params = {
        "pubStartDate": start_date.strftime("%Y-%m-%dT%H:%M:%S.000"),
        "pubEndDate": end_date.strftime("%Y-%m-%dT%H:%M:%S.000"),
        "resultsPerPage": 2000,
        "startIndex": 0
    }

    headers = {}
    if API_KEY:
        headers["apiKey"] = API_KEY

    all_cves = []

    print("Fetching CVEs from NVD...")
    print(f"Date range: {start_date} to {end_date}")

    while True:
        try:
            response = requests.get(
                API_URL,
                params=params,
                headers=headers,
                timeout=30
            )
            response.raise_for_status()

            data = response.json()
            vulnerabilities = data.get("vulnerabilities", [])

            all_cves.extend(vulnerabilities)

            total = data.get("totalResults", 0)
            print(f"Retrieved {len(all_cves)} / {total} CVEs")

            if len(all_cves) >= total or not vulnerabilities:
                break

            params["startIndex"] += len(vulnerabilities)

            # Respect NVD public API rate limits
            time.sleep(6)

        except requests.exceptions.RequestException as error:
            print(f"API request failed: {error}")
            return []

    results = []

    for item in all_cves:
        cve = item.get("cve", {})

        description = next(
            (
                desc["value"]
                for desc in cve.get("descriptions", [])
                if desc.get("lang") == "en"
            ),
            "No description available"
        )

        score, severity = get_cvss(cve)

        results.append({
            "CVE ID": cve.get("id"),
            "Published": cve.get("published"),
            "Last Modified": cve.get("lastModified"),
            "Severity": severity,
            "CVSS Score": score,
            "Description": description,
            "Status": cve.get("vulnStatus"),
            "URL": f"https://nvd.nist.gov/vuln/detail/{cve.get('id')}"
        })

    return results


def main():
    cves = scrape_cves()

    if not cves:
        print("No CVEs found or API request failed.")
        return

    df = pd.DataFrame(cves)

    # Sort newest CVEs first
    df = df.sort_values("Published", ascending=False)

    # Export to CSV
    filename = "recent_cves.csv"
    df.to_csv(filename, index=False)

    print("\n" + "=" * 60)
    print(f"TOTAL CVEs FOUND: {len(df)}")
    print("=" * 60)

    for _, row in df.iterrows():
        print(f"\nCVE ID: {row['CVE ID']}")
        print(f"Severity: {row['Severity']}")
        print(f"CVSS Score: {row['CVSS Score']}")
        print(f"Published: {row['Published']}")
        print(f"Description: {row['Description'][:200]}...")
        print(f"URL: {row['URL']}")
        print("-" * 60)

    print(f"\nResults saved to: {filename}")


if __name__ == "__main__":
    main()
